# Считаю retrieval метрики по DINOv2 

In [1]:
import polars as pl
import numpy as np
from pathlib import Path
import chromadb

In [2]:
PROJECT_ROOT_PATH = Path('/Users/alexansemyachkin/Desktop/studies/hse/sneakers-hse')

In [3]:
df = pl.read_parquet(PROJECT_ROOT_PATH / "data/04_dinov2_embeddings.parquet.gzip")

embeddings = np.stack(df["embedding"].to_list()).astype("float32")
labels = np.array(df["class"].to_list())
paths = df["path"].to_list()
df['class'].value_counts()

class,count
str,u32
"""Vans Кеды Upland""",275
"""Nike Кроссовки Nike Zoom Vomer…",282
"""Tendance Кеды""",269
"""Under Armour Кроссовки UA Char…",281
"""Vans Кеды Sport Low""",278
…,…
"""Kari Кроссовки""",273
"""Reebok Кеды CLUB C 85""",283
"""PUMA Кроссовки Darter Pro""",289


In [4]:
client = chromadb.Client()
collection = client.get_or_create_collection(
    "embeddings",
    metadata={"hnsw:space": "cosine"})

def add_in_batches(collection, ids, embeddings, metadatas, batch_size=5000):
    for i in range(0, len(ids), batch_size):
        collection.add(
            ids=ids[i:i+batch_size],
            embeddings=embeddings[i:i+batch_size],
            metadatas=metadatas[i:i+batch_size]
        )

add_in_batches(
    collection,
    ids=paths,
    embeddings=embeddings,
    metadatas=[{"class": c} for c in labels]
)

In [5]:
from tqdm.notebook import tqdm

def get_neighbors(collection, embeddings, k=10, batch_size=10):
    all_results = []
    for i in tqdm(range(0, len(embeddings), batch_size), desc="Querying"):
        batch = embeddings[i:i+batch_size]
        results = collection.query(
            query_embeddings=batch,
            n_results=k + 1  # +1 чтобы убрать self
        )['metadatas']
        all_results.extend(results)
    return np.array([[neighbor['class'] for neighbor in query]
                     for query in all_results])[:, 1:]  # Убираю self-match

def hit_rate_at_k(neighbors, labels, k):
    hits = 0
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        if query_label in retrieved:
            hits += 1
    return hits / len(labels)

def precision_at_k(neighbors, labels, k):
    precisions = []
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        relevant = (retrieved == query_label).sum()
        precisions.append(relevant / k)
    return np.mean(precisions)

def average_precision(retrieved, query_label, k):
    score = 0.0
    hits = 0
    for i in range(k):
        if retrieved[i] == query_label:
            hits += 1
            score += hits / (i + 1)
    return score / hits if hits > 0 else 0


def map_at_k(neighbors, labels, k):
    scores = []
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        scores.append(average_precision(retrieved, query_label, k))
    return np.mean(scores)

neighbors = get_neighbors(collection, embeddings, k=20)
len(neighbors[0])
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    print("HitRate:", hit_rate_at_k(neighbors, labels, k))
    print("Precision:", precision_at_k(neighbors, labels, k))
    print("MAP:", map_at_k(neighbors, labels, k))

Querying:   0%|          | 0/1127 [00:00<?, ?it/s]


Metrics @ 1
HitRate: 0.7297825122059476
Precision: 0.7297825122059476
MAP: 0.7297825122059476

Metrics @ 5
HitRate: 0.8919662671992898
Precision: 0.6078118064802486
MAP: 0.7621949746017657

Metrics @ 10
HitRate: 0.9345761207279183
Precision: 0.5364225477141589
MAP: 0.7185945699398926


In [6]:
df = df.with_columns(neighbors=neighbors)
df = df.with_columns(
    pl.struct(["neighbors", "class"]).map_elements(
        lambda row: [
            1 if n == row["class"] else 0
            for n in row["neighbors"][:k]
        ]
    ).alias("hit_flg")
)
df

path,class,embedding,neighbors,hit_flg
str,str,list[f64],"array[str, 20]",list[i64]
"""Vans Кеды Upland/clothing_0_26…","""Vans Кеды Upland""","[-1.930393, -0.157552, … 0.300103]","[""Vans Кеды Hylane"", ""PUMA Кроссовки RS-X Efekt PRM"", … ""PUMA Кроссовки Darter Pro""]","[0, 0, … 0]"
"""Vans Кеды Upland/clothing_0_57…","""Vans Кеды Upland""","[-1.749115, -1.480964, … -0.966375]","[""Vans Кеды Upland"", ""Vans Кеды Upland"", … ""Vans Кеды Upland""]","[1, 1, … 0]"
"""Vans Кеды Upland/orig_45.jpeg""","""Vans Кеды Upland""","[0.503809, -0.948843, … -3.250608]","[""Nike Кеды NIKE AIR FORCE 1"", ""PUMA Кеды CA Pro Classic II"", … ""Nike Кеды NIKE AIR FORCE 1""]","[0, 0, … 1]"
"""Vans Кеды Upland/clothing_0_0.…","""Vans Кеды Upland""","[-1.441995, -2.107342, … -0.808429]","[""Vans Кеды Upland"", ""Vans Кеды Upland"", … ""PUMA Кеды Palermo""]","[1, 1, … 0]"
"""Vans Кеды Upland/clothing_0_23…","""Vans Кеды Upland""","[-0.845633, -2.200308, … -1.242736]","[""Vans Кеды Hylane"", ""Vans Кеды Upland"", … ""Vans Кеды Hylane""]","[0, 1, … 1]"
…,…,…,…,…
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[0.131167, -0.318168, … -0.66955]","[""Under Armour Кроссовки UA Phantom 4"", ""Under Armour Кроссовки UA Phantom 4"", … ""Under Armour Кроссовки UA Charged Rogue 5""]","[1, 1, … 0]"
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.267569, -3.275375, … -2.853185]","[""Under Armour Кроссовки UA Phantom 4"", ""Under Armour Кроссовки UA Phantom 4"", … ""Under Armour Кроссовки UA Phantom 4""]","[1, 1, … 1]"
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.495174, -1.613851, … -1.374945]","[""Under Armour Кроссовки UA Phantom 4"", ""Under Armour Кроссовки UA Phantom 4"", … ""Under Armour Кроссовки UA Phantom 4""]","[1, 1, … 1]"


In [7]:
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    df_k = df.with_columns(hits_k = pl.col("hit_flg").list.slice(0, k))\
        .with_columns(
            pos = pl.int_ranges(1, pl.col("hits_k").list.len() + 1),
            hit_at_k = pl.col("hits_k").list.max(),
            precision_at_k = pl.col("hits_k").list.mean()
        )
    display(df_k.select(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))
    display(df_k.group_by('class').agg(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))


Metrics @ 1


hit_at_k,precision_at_k
f64,f64
0.729783,0.729783


class,hit_at_k,precision_at_k
str,f64,f64
"""X-Plode Кеды""",0.551852,0.551852
"""Maison David Кроссовки""",0.77972,0.77972
"""Tendance Кроссовки""",0.505535,0.505535
"""PUMA Кроссовки Hypnotic LS""",0.794326,0.794326
"""Vans Кеды Old Skool""",0.797909,0.797909
…,…,…
"""adidas Кроссовки DURAMO SL2 M""",0.493952,0.493952
"""adidas Кроссовки GALAXY 7 M""",0.777778,0.777778
"""adidas Кеды VL COURT 3""",0.648485,0.648485



Metrics @ 5


hit_at_k,precision_at_k
f64,f64
0.891966,0.607812


class,hit_at_k,precision_at_k
str,f64,f64
"""Maison David Кеды""",0.922509,0.622878
"""Under Armour Кроссовки UA Char…",0.915493,0.584507
"""X-Plode Кроссовки""",0.713755,0.327138
"""Nike Кеды Dunk Low Retro""",0.965157,0.777003
"""Reebok Кроссовки CLASSIC LEATH…",0.897887,0.567606
…,…,…
"""Under Armour Кроссовки UA Phan…",0.925267,0.730249
"""PUMA Кроссовки Puma Morphic""",0.954198,0.803817
"""PUMA Кроссовки Darter Pro""",0.948097,0.802768



Metrics @ 10


hit_at_k,precision_at_k
f64,f64
0.934576,0.536423


class,hit_at_k,precision_at_k
str,f64,f64
"""adidas Кроссовки GALAXY 7 M""",0.96875,0.603472
"""Reebok Кеды CLUB C GROUNDS UK""",0.938849,0.648561
"""Vans Кеды Upland""",0.949091,0.522182
"""Kari Кроссовки""",0.912088,0.478388
"""Nike Кроссовки Nike Zoom Vomer…",0.98227,0.786879
…,…,…
"""Under Armour Кроссовки UA CHAR…",0.95122,0.375261
"""Nike Кроссовки AIR MAX 90""",0.965157,0.789895
"""Reebok Кроссовки CLASSIC LEATH…",0.929577,0.482042
